# 03 — BPE Merge Trace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/03_bpe_merge_trace.ipynb)

Watch byte-pair encoding learn merges one step at a time on a tiny corpus.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
%pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
%pip install -q -e .

import sys
src = '/content/forge-tokenizer/src'
if src not in sys.path:
    sys.path.insert(0, src)
print('Ready. Run the Imports cell next.')

## 2. Imports

In [ ]:
from pathlib import Path
assert Path('src/forge_tokenizer').exists(), 'Run section 1: Install & Clone first.'
from collections import Counter

from forge_tokenizer.bpe import get_pair_counts, merge_pair
from forge_tokenizer.pretokenizer import pretokenize

print('forge-tokenizer ready')

## 3. Tiny Corpus

The spaces are intentional. A byte-level tokenizer must learn that leading spaces are meaningful.

In [ ]:
toy_text = ' low lower lowest low low newest widest'
pre_tokens = pretokenize(toy_text)
word_freqs = Counter(pre_tokens)

print(pre_tokens)
print(word_freqs)

## 4. Start From UTF-8 Bytes

In [ ]:
def byte_symbols(piece):
    return tuple(bytes([b]) for b in piece.encode('utf-8'))

corpus_tokens = {byte_symbols(piece): freq for piece, freq in word_freqs.items()}
for symbols, freq in corpus_tokens.items():
    print(freq, symbols)

## 5. Count Adjacent Pairs

BPE repeatedly merges the most frequent adjacent pair. Ties are broken deterministically by byte order in this repository.

In [ ]:
pair_counts = get_pair_counts(corpus_tokens)
for pair, count in pair_counts.most_common(10):
    print(pair, count)

## 6. Run The First 8 Merge Steps

In [ ]:
trace = []
working = corpus_tokens

for step in range(8):
    pair_counts = get_pair_counts(working)
    best_pair, count = min(pair_counts.items(), key=lambda item: (-item[1], item[0][0], item[0][1]))
    trace.append((step, best_pair, count, best_pair[0] + best_pair[1]))
    working = merge_pair(best_pair, working)

for step, pair, count, merged in trace:
    print(f'{step:02d}: {pair[0]!r} + {pair[1]!r} -> {merged!r}  count={count}')

## 7. Final Segmentation After Those Merges

In [ ]:
for symbols, freq in working.items():
    decoded = [symbol.decode('utf-8', errors='replace') for symbol in symbols]
    print(f'{freq}x', decoded)

---

**Next →** [04 Token Tax Lab](04_token_tax_lab.ipynb)